# Session 5 – RAG: why, how, and where it breaks
Seven published papers on immunotherapy for mismatch-repair-deficient (dMMR) endometrial cancer, searched by a small RAG pipeline you run step by step.
Everything sent to the model is public (Tier 1). The papers must be in a folder called `papers` next to this notebook.

In [ ]:
import os, sys, re, glob, json, time, getpass, subprocess
for pkg, mod in [("openai", "openai"), ("pypdf", "pypdf"), ("scikit-learn", "sklearn"), ("numpy", "numpy")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import numpy as np, pypdf
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Connect
The same NVIDIA key as in Session 4. It is not shown or saved.

In [ ]:
API_KEY = os.environ.get("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API key: ")
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=API_KEY)

MODEL = None                                   # or set by hand, e.g. MODEL = "google/gemma-4-31b-it"
CANDIDATES = ["google/gemma-4-31b-it", "nvidia/nemotron-3.5-lightning-30b-a3b", "openai/gpt-oss-20b",
              "nvidia/nemotron-3-super-120b-a12b", "moonshotai/kimi-k3", "z-ai/glm-5.3"]
EMBED_MODEL = None                             # the model that turns text into meaning-vectors
EMBED_CANDIDATES = ["nvidia/llama-nemotron-embed-vl-1b-v2", "nvidia/llama-3.2-nv-embedqa-1b-v2", "nvidia/nv-embedqa-e5-v5", "baai/bge-m3"]

def works(m):
    try:
        t = time.time()
        client.chat.completions.create(model=m, messages=[{"role": "user", "content": "Hi"}], max_tokens=5)
        print(f"  {m}: OK ({time.time() - t:.1f} s)")
        return True
    except Exception as e:
        print(f"  {m}: not available ({str(e)[:60]})")
        return False

LOCAL = None                                   # local embedding model, used only if NVIDIA has none

def embed(texts, kind="passage", model=None):
    model = model or EMBED_MODEL
    if model.startswith("local:"):
        return LOCAL.encode(texts, normalize_embeddings=True, batch_size=32)
    out = []
    for i in range(0, len(texts), 50):                       # 50 texts per request
        try:
            r = client.embeddings.create(model=model, input=texts[i:i + 50],
                                         extra_body={"input_type": kind, "truncate": "END"})
        except Exception as first_error:
            try:
                r = client.embeddings.create(model=model, input=texts[i:i + 50])
            except Exception:
                raise first_error
        out += [d.embedding for d in r.data]
    v = np.array(out, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def embed_works(m):
    try:
        embed(["test"], "query", model=m)
        print(f"  {m}: OK")
        return True
    except Exception as e:
        print(f"  {m}: not available ({str(e)[:150]})")
        return False

MODEL = MODEL or next((m for m in CANDIDATES if works(m)), None)
if EMBED_MODEL is None:
    EMBED_MODEL = next((m for m in EMBED_CANDIDATES if embed_works(m)), None)
if EMBED_MODEL is None:                                   # anything NVIDIA currently lists as an embedding model
    try:
        listed = [m.id for m in client.models.list().data if "embed" in m.id and m.id not in EMBED_CANDIDATES]
        print("  NVIDIA lists:", listed)
        EMBED_MODEL = next((m for m in listed if embed_works(m)), None)
    except Exception as e:
        print("  could not list models:", str(e)[:150])
if EMBED_MODEL is None:                                   # last resort: a small model on this server
    try:
        from sentence_transformers import SentenceTransformer
        LOCAL = SentenceTransformer("BAAI/bge-small-en-v1.5")
        EMBED_MODEL = "local:bge-small-en-v1.5"
    except Exception as e:
        print("  no local model either:", str(e)[:150])
print("Answers:", MODEL, "| Meaning search:", EMBED_MODEL)

In [ ]:
def ask(question, system="", max_tokens=400):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    t = time.time()
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.2, max_tokens=max_tokens)
    return r.choices[0].message.content, time.time() - t

def ask_many(jobs):                                    # several questions at the same time
    with ThreadPoolExecutor(max_workers=len(jobs)) as ex:
        return list(ex.map(lambda j: ask(*j), jobs))

## 2. Usage – the same question, without and with the papers

In [ ]:
NO_PAPERS = "You are a careful medical research assistant. Answer briefly and give your sources."

USAGE_Q = "In the RUBY trial, how much did adding dostarlimab to chemotherapy improve progression-free survival in patients with dMMR endometrial cancer?"
(no_papers_answer, t), = ask_many([(USAGE_Q, NO_PAPERS)])
print(f"WITHOUT PAPERS ({t:.0f} s)\n{no_papers_answer}")

Now with the papers. First, the pipeline: **papers → text → chunks → search → top passages → model → answer with citations.**

In [ ]:
# each paper is recognised by a phrase on its first page, whatever the file is called
MARKERS = [("Le 2017", "PD-1 blockade"), ("KEYNOTE-158 2022", "KEYNOTE-158"), ("GARNET 2022", "jitc-2021-003777"),
           ("Chow 2023", "Two Modes of Response"), ("NRG-GY018 2023", "NEJMoa2302312"),
           ("RUBY 2023", "NEJMoa2216334"), ("NRG-GY018 OS 2025", "s41591-025-03566-1")]
NAMES = {n: n for n, m in MARKERS}

def clean(t):
    t = re.sub(r"(\w) ?[-\u00ad] ?\n\s*(\w)", r"\1\2", t)      # re-join words split across lines
    return re.sub(r"\s+", " ", t).strip()

# papers -> text (one entry per page)
pdfs = sorted(glob.glob("papers/**/*.pdf", recursive=True))      # also finds papers/papers/*.pdf after unzipping
print(len(pdfs), "PDFs found:", [os.path.basename(f) for f in pdfs])
assert len(pdfs) == 7, "Expected 7 PDFs in a folder called 'papers' next to this notebook"
pages = []
for f in pdfs:
    texts = [clean(p.extract_text() or "") for p in pypdf.PdfReader(f).pages]
    name = next((n for n, m in MARKERS if m.lower() in texts[0].lower()), os.path.basename(f)[:-4])
    print(f"  {os.path.basename(f)} -> {name}")
    pages += [{"paper": name, "page": i, "text": t} for i, t in enumerate(texts, 1)]

# text -> chunks of about 150 words, cut at sentence ends
CHUNK_WORDS = 150
chunks = []
for p in pages:
    buf = []
    for s in re.split(r"(?<=[.!?])\s+(?=[A-Z(])", p["text"]):
        buf.append(s)
        if sum(len(x.split()) for x in buf) >= CHUNK_WORDS:
            chunks.append({"paper": p["paper"], "page": p["page"], "text": " ".join(buf)}); buf = []
    if sum(len(x.split()) for x in buf) > 20:
        chunks.append({"paper": p["paper"], "page": p["page"], "text": " ".join(buf)})
for i, c in enumerate(chunks):
    c["id"] = i

print(len(pages), "pages ->", len(chunks), "chunks")
for name in NAMES.values():
    print(f"  {name:<20} {sum(c['paper'] == name for c in chunks):>4} chunks")
example = next(c for c in chunks if "analyzed in two independent cohorts" in c["text"])
print("\nExample chunk:", example["paper"], "p.", example["page"], "\n", example["text"][:600])

In [ ]:
# chunks -> two search indexes
# 1) keyword search: scores chunks by the words they share with the question
tfidf = TfidfVectorizer(stop_words="english", token_pattern=r"(?u)\b[\w-]+\b")
KW = tfidf.fit_transform([c["text"] for c in chunks])

# 2) meaning search: each chunk becomes a vector of numbers; similar meaning -> similar vectors
CACHE = f"{os.path.dirname(pdfs[0])}/embeddings_{str(EMBED_MODEL).replace('/', '_').replace(':', '_')}.npy"
if EMBED_MODEL is None:
    EMB = None
    print("No meaning search available: keyword search only (see the Connect cell for the reason)")
elif os.path.exists(CACHE) and len(np.load(CACHE)) == len(chunks):
    EMB = np.load(CACHE)
else:
    t = time.time()
    EMB = embed([c["text"] for c in chunks], "passage")
    np.save(CACHE, EMB)
    print(f"Embedded {len(chunks)} chunks in {time.time() - t:.0f} s (saved for next time)")
print("Index ready:", KW.shape[0], "chunks")

def search(question, k=5, mode="meaning", exclude=()):
    if mode == "keyword" or EMB is None:
        scores = cosine_similarity(tfidf.transform([question]), KW)[0]
    else:
        scores = EMB @ embed([question], "query")[0]
    order = [i for i in scores.argsort()[::-1] if chunks[i]["paper"] not in exclude]
    return [(chunks[i], float(scores[i])) for i in order[:k]]

def show(hits, width=160):
    for n, (c, s) in enumerate(hits, 1):
        print(f"[{n}] {c['paper']}, p.{c['page']} (score {s:.2f}): {c['text'][:width]}...")

In [ ]:
RAG_RULES = """You answer questions using ONLY the numbered passages below, taken from published papers.
- Cite the passage number after every claim, like [2].
- If the passages do not contain the answer, say "The papers provided do not answer this." Do not use outside knowledge.
- Say which paper and year a number comes from."""

def rag_jobs(question, k=5, mode="meaning", exclude=()):
    hits = search(question, k, mode, exclude)
    context = "\n\n".join(f"[{n}] ({c['paper']}, p.{c['page']}) {c['text']}" for n, (c, s) in enumerate(hits, 1))
    return hits, (f"PASSAGES:\n{context}\n\nQUESTION: {question}", RAG_RULES)

def rag(question, k=5, mode="meaning", exclude=(), quiet=False):
    hits, job = rag_jobs(question, k, mode, exclude)
    answer, t = ask(*job)
    if not quiet:
        print(f"Q: {question}\n\nRETRIEVED ({mode}, top {k}):"); show(hits)
        print(f"\nANSWER ({t:.0f} s):\n{answer}")
    return answer, hits

def check_citations(answer, hits):
    """Each cited sentence next to the full passage it cites: does the passage say that?"""
    for sent in re.split(r"(?<=[.!?])\s+", answer):
        for n in sorted(set(int(x) for x in re.findall(r"\[(\d+)\]", sent))):
            if 1 <= n <= len(hits):
                c = hits[n - 1][0]
                print(f"CLAIM: {sent.strip()}\nPASSAGE [{n}] {c['paper']}, p.{c['page']}:\n{c['text']}\n{'-' * 80}")

In [ ]:
answer, hits = rag(USAGE_Q)
print("\n" + "=" * 80 + "\nWITHOUT PAPERS:\n" + no_papers_answer)

## 3. Limitations
Each cell tests one way RAG fails. Read what was **retrieved**, not only the answer. Record each result on the shared sheet:
correct? · does the cited passage support it? · did it say "not in the papers" when it should?

### L1 – The right passage is never found
The papers do not all use the same words: Chow writes *MMRd*, Le writes *mismatch repair deficient*. Which search finds Le 2017, whose title answers this question?

In [ ]:
Q1 = "Does dMMR status predict response to immunotherapy?"
for mode in ["keyword", "meaning"]:
    hits = search(Q1, mode=mode)
    print(f"--- {mode} search ---"); show(hits)
    print("Le 2017 found:", any(c["paper"] == "Le 2017" for c, s in hits), "\n")
for name in NAMES.values():
    n = sum(bool(re.search(r"\bdMMR\b", c["text"])) for c in chunks if c["paper"] == name)
    print(f"chunks containing 'dMMR' in {name:<20}: {n}")

### L2 – A similar but wrong passage
dMMR and pMMR differ by one letter and have opposite meanings. Check that each number belongs to the right group.

In [ ]:
answer2, hits2 = rag("In NRG-GY018, what was the hazard ratio for progression-free survival in the pMMR cohort?")

### L3 – Chunking cuts the meaning
The pipeline cuts text every ~150 words and at every page break. What is lost at the cut? What happened to the table?

In [ ]:
le = next((c for c in chunks if c["paper"] == "Le 2017" and "originating in other organs" in c["text"]), None)
if le:
    print("END OF ONE CHUNK:  ...", chunks[le["id"] - 1]["text"][-220:])
    print("\nSTART OF THE NEXT:", le["text"][:220], "...")
table = [c for c in chunks if c["paper"] == "RUBY 2023" and "pMMR–MSS 0 0" in c["text"]]
print("\nA TABLE, AFTER PDF EXTRACTION:\n", table[0]["text"][:600] if table else "(not found)")

### L4 – Top-k is too small
A question that needs all seven papers, answered from the best 5 passages, then from 15. Which papers are missing, and does the first answer admit it?

In [ ]:
Q4 = "Summarize the objective response rates to PD-1 blockade reported across all the papers."
(h5, j5), (h15, j15) = rag_jobs(Q4, k=5), rag_jobs(Q4, k=15)
(a5, t5), (a15, t15) = ask_many([j5, j15])
for k, h, a in [(5, h5, a5), (15, h15, a15)]:
    print(f"=== top {k}: papers used = {sorted({c['paper'] for c, s in h})}\n{a}\n")

### L5 – Does the citation support the claim?
Every cited sentence from the top-15 answer, next to the passage it cites.

In [ ]:
check_citations(a15, h15)

### L6 – The answer is not in the papers
AtTEnd is not in the collection. DUO-E is only mentioned inside another paper's discussion.

In [ ]:
Q6a = "What did the AtTEnd trial show for atezolizumab plus chemotherapy in dMMR endometrial cancer?"
Q6b = "What did the DUO-E trial show?"
(h6a, j6a), (h6b, j6b) = rag_jobs(Q6a), rag_jobs(Q6b)
(a6a, _), (a6b, _) = ask_many([j6a, j6b])
for q, h, a in [(Q6a, h6a, a6a), (Q6b, h6b, a6b)]:
    print(f"Q: {q}\nRETRIEVED:"); show(h, 120); print(f"ANSWER:\n{a}\n{'=' * 80}")

### L7 – Old and new evidence
NRG-GY018 was published in 2023 and updated in 2025. Does the answer say which result is newer?

In [ ]:
answer7, hits7 = rag("What was the hazard ratio for progression-free survival in the dMMR cohort of NRG-GY018?")

### L8 – Who picked the papers?
The same question with all papers, then without Chow 2023.

In [ ]:
Q8 = "Does the cause of mismatch-repair deficiency (inherited or mutational versus MLH1 promoter methylation) affect response to PD-1 blockade?"
(h8, j8), (h8x, j8x) = rag_jobs(Q8), rag_jobs(Q8, exclude=("Chow 2023",))
(a8, _), (a8x, _) = ask_many([j8, j8x])
print(f"=== ALL PAPERS ===\n{a8}\n\n=== WITHOUT Chow 2023 ===\n{a8x}")

## 4. Your own test
Find one more question that breaks it. Change the question, `k`, `mode` or `exclude`.

In [ ]:
answer, hits = rag("Your question", k=5, mode="meaning")